In [ ]:
# movie_recommender.py
import os
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib


DATA_PATH = os.path.join("data", "movies.csv")   # path to your movies CSV
VECT_PATH = os.path.join("model", "tfidf_vectorizer.joblib")
SIM_PATH = os.path.join("model", "cosine_sim_matrix.joblib")
INDEX_PATH = os.path.join("model", "indices.joblib")

def load_data(path=DATA_PATH):
    """Load movies dataset into a DataFrame."""
    df = pd.read_csv(path)
    return df

def preprocess(df):
    """
    Create a 'metadata' column that concatenates text fields we want to use
    for content-based recommendations (e.g., genres + overview).
    """

    df = df.copy()
    # Ensure columns exist and convert to string to avoid errors
    # Prefer overview if available, otherwise use genres
    # We'll combine both if both are present.
    # Safe-get columns:
    overview = df['overview'] if 'overview' in df.columns else pd.Series([''] * len(df))
    genres = df['genres'] if 'genres' in df.columns else pd.Series([''] * len(df))
    # Replace NaN with empty string
    overview = overview.fillna('').astype(str)
    genres = genres.fillna('').astype(str)
    # Create a combined metadata column
    df['metadata'] = genres + ' ' + overview
    # Optional: basic cleaning - lowercasing
    df['metadata'] = df['metadata'].str.lower()
    # Keep only relevant columns for downstream steps
    # We'll need title and metadata and original index
    df = df.reset_index(drop=True)
    return df

def build_tfidf_matrix(df, max_features=5000):
    """
    Build TF-IDF matrix from the 'metadata' column.
    Returns the fitted vectorizer and the TF-IDF sparse matrix.
    """
    tfidf = TfidfVectorizer(stop_words='english', max_features=max_features)
    tfidf_matrix = tfidf.fit_transform(df['metadata'])
    return tfidf, tfidf_matrix

def build_similarity(tfidf_matrix):
    """Compute cosine similarity matrix from TF-IDF vectors."""
    # 28
    cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
    # 29
    return cosine_sim

def create_indices(df):
    """
    Create a mapping (title -> index) for fast lookup.
    Titles are normalized (lowercased) to make lookup case-insensitive.
    """
    indices = pd.Series(df.index, index=df['title'].str.lower()).to_dict()
    return indices

def save_artifacts(tfidf, cosine_sim, indices, vect_path=VECT_PATH, sim_path=SIM_PATH, idx_path=INDEX_PATH):
    """Save vectorizer, similarity matrix, and indices for later use."""
    os.makedirs(os.path.dirname(vect_path), exist_ok=True)
    joblib.dump(tfidf, vect_path)
    joblib.dump(cosine_sim, sim_path)
    joblib.dump(indices, idx_path)

def load_artifacts(vect_path=VECT_PATH, sim_path=SIM_PATH, idx_path=INDEX_PATH):
    """Load artifacts if they exist; otherwise return None."""
    if not os.path.exists(vect_path) or not os.path.exists(sim_path) or not os.path.exists(idx_path):
        return None, None, None

    tfidf = joblib.load(vect_path)
    cosine_sim = joblib.load(sim_path)
    indices = joblib.load(idx_path)
    return tfidf, cosine_sim, indices

def get_recommendations(title, df, cosine_sim, indices, top_n=10):
    """
    Given a movie title, return top_n similar movies based on cosine similarity.
    """
    title_lower = title.lower()
    if title_lower not in indices:
        raise ValueError(f"Title '{title}' not found in the dataset.")
    idx = indices[title_lower]
    sim_scores = list(enumerate(cosine_sim[idx]))
    # Sort movies by similarity score in descending order
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    # Skip the first one because it is the movie itself (score 1.0)
    sim_scores = sim_scores[1: top_n + 1]
    movie_indices = [i[0] for i in sim_scores]
    return df.iloc[movie_indices][['title', 'metadata']].copy()

def train_and_build(path=DATA_PATH):
    """High-level function to train and build artifacts from raw CSV."""
    df = load_data(path)
    df = preprocess(df)
    tfidf, tfidf_matrix = build_tfidf_matrix(df)
    cosine_sim = build_similarity(tfidf_matrix)
    indices = create_indices(df)
    save_artifacts(tfidf, cosine_sim, indices)
    return df, tfidf, cosine_sim, indices

if __name__ == "__main__":
    # Train and build artifacts (only if not already saved)
    tfidf, cosine_sim, indices = load_artifacts()

    if tfidf is None:
        print("Training TF-IDF and building similarity matrix...")
        df, tfidf, cosine_sim, indices = train_and_build()
    else:
        print("Loaded saved artifacts.")
        df = preprocess(load_data())

    # Simple interactive prompt for quick testing
    while True:

        query = input("\nEnter a movie title (or 'exit' to quit): ").strip()

        if query.lower() == 'exit':
            print("Bye!")
            break

        try:
            recs = get_recommendations(query, df, cosine_sim, indices, top_n=10)
            print(f"\nTop recommendations for '{query}':\n")
            for i, row in enumerate(recs.itertuples(index=False), start=1):
                print(f"{i}. {row.title}")
                
        except ValueError as e:
            print(e)
